# Data Science Hands-On Lab — Week 2 Internship
**Internship organization:** Yuva  
**Intern:** [Your Full Name]  
**Week:** 2  
**Topic:** Python environment, data handling, cleaning, visualization, and exploratory data analysis

## Objective
This notebook demonstrates a reproducible data-analysis workflow using Python, NumPy, Pandas, Matplotlib, Seaborn, and Scikit-learn. The Iris dataset is used as a small, well-known sample dataset. Run the cells from top to bottom to generate the outputs and charts.

> Before submission: replace the name field, run all cells, review the outputs, and add screenshots if your internship supervisor requires them.

## 1. Set up and verify the environment
If a library is missing, install it in a terminal or notebook using `%pip install numpy pandas matplotlib seaborn scikit-learn jupyter`. In a managed internship environment, follow the organization's installation policy.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Seaborn:", sns.__version__)

## 2. Load a dataset and inspect it
The Iris dataset contains measurements of iris flowers and a species label. It is used here for demonstration and does not represent real customer or business data.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.copy()
df = df.rename(columns={
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
    "target": "species_id"
})
df["species"] = df["species_id"].map(dict(enumerate(iris.target_names)))

display(df.head(10))
print("Shape (rows, columns):", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nSummary statistics:")
display(df.describe().T)

## 3. Pandas selection, filtering, sorting, and aggregation

In [ ]:
# Select columns
display(df[["sepal_length", "petal_length", "species"]].head())

# Filter rows: petal length above 4.5 cm and species is virginica
filtered = df[(df["petal_length"] > 4.5) & (df["species"] == "virginica")]
print("Filtered row count:", len(filtered))
display(filtered.head())

# Sort by petal length from largest to smallest
display(df.sort_values("petal_length", ascending=False).head(10))

# Group by species and aggregate
species_summary = df.groupby("species").agg(
    average_sepal_length=("sepal_length", "mean"),
    maximum_petal_width=("petal_width", "max"),
    petal_length_std=("petal_length", "std"),
    flower_count=("species", "size")
).round(3)
display(species_summary)

## 4. NumPy numerical operations and transformations

In [ ]:
petal_lengths = df["petal_length"].to_numpy()
print("Mean petal length:", round(np.mean(petal_lengths), 3))
print("Median petal length:", round(np.median(petal_lengths), 3))
print("Standard deviation:", round(np.std(petal_lengths), 3))
print("Minimum and maximum:", np.min(petal_lengths), np.max(petal_lengths))

# Min-max scaling of a numeric column (creates a new series; original data is unchanged)
df["petal_length_scaled"] = (
    (df["petal_length"] - df["petal_length"].min()) /
    (df["petal_length"].max() - df["petal_length"].min())
)
display(df[["petal_length", "petal_length_scaled"]].head())

## 5. Data cleaning and data-quality checks
The Iris dataset is generally clean. To demonstrate cleaning techniques safely, a separate copy is deliberately modified with one missing value and one duplicate row. The original `df` remains unchanged.

In [ ]:
dirty = df.drop(columns=["petal_length_scaled"]).copy()
dirty.loc[0, "sepal_length"] = np.nan
dirty = pd.concat([dirty, dirty.iloc[[1]]], ignore_index=True)

print("Missing values before cleaning:")
display(dirty.isna().sum().to_frame("missing_count"))
print("Duplicate rows before cleaning:", dirty.duplicated().sum())

clean = dirty.drop_duplicates().copy()
clean["sepal_length"] = clean["sepal_length"].fillna(clean["sepal_length"].median())

# Standardize text labels in case inconsistent capitalization/spacing appears
clean["species"] = clean["species"].astype("string").str.strip().str.lower()

print("Missing values after cleaning:")
display(clean.isna().sum().to_frame("missing_count"))
print("Duplicate rows after cleaning:", clean.duplicated().sum())
print("Cleaned shape:", clean.shape)

### Data-cleaning observations
- Missing values were identified with `isna()` and filled using the median of the affected numeric column.
- Duplicate rows were identified with `duplicated()` and removed with `drop_duplicates()`.
- Text labels were normalized with `str.strip().str.lower()`.
- In a real project, the choice to drop, fill, or retain records should depend on the meaning of the data and be documented.

## 6. Data visualization

In [ ]:
sns.set_theme()

# Bar chart: average petal length by species
avg_petal = df.groupby("species", as_index=False)["petal_length"].mean()
plt.figure(figsize=(7, 4))
sns.barplot(data=avg_petal, x="species", y="petal_length")
plt.title("Average Petal Length by Species")
plt.xlabel("Species")
plt.ylabel("Average petal length (cm)")
plt.tight_layout()
plt.show()

In [ ]:
# Histogram: distribution of sepal length
plt.figure(figsize=(7, 4))
sns.histplot(data=df, x="sepal_length", bins=15, kde=True)
plt.title("Distribution of Sepal Length")
plt.xlabel("Sepal length (cm)")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: petal length versus petal width
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df, x="petal_length", y="petal_width", hue="species")
plt.title("Petal Length vs Petal Width")
plt.tight_layout()
plt.show()

In [ ]:
# Box plot: petal length by species
plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="species", y="petal_length")
plt.title("Petal Length Spread by Species")
plt.tight_layout()
plt.show()

In [ ]:
# Line chart: first 30 observations, shown in dataset order
sample = df.iloc[:30].reset_index(names="observation")
plt.figure(figsize=(8, 4))
sns.lineplot(data=sample, x="observation", y="sepal_length")
plt.title("Sepal Length Across the First 30 Observations")
plt.xlabel("Observation index")
plt.ylabel("Sepal length (cm)")
plt.tight_layout()
plt.show()

## 7. Descriptive statistics, relationships, and correlations

In [ ]:
numeric_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
display(df[numeric_cols].describe().T)

corr = df[numeric_cols].corr(numeric_only=True)
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation Matrix of Flower Measurements")
plt.tight_layout()
plt.show()

print("Correlation between petal length and petal width:",
      round(df["petal_length"].corr(df["petal_width"]), 3))

## 8. Merge and transformation practice
This exercise creates a small lookup table and merges it with the dataset to demonstrate a common Pandas workflow.

In [ ]:
species_lookup = pd.DataFrame({
    "species": ["setosa", "versicolor", "virginica"],
    "species_group": ["Group A", "Group B", "Group C"]
})
merged = df.merge(species_lookup, on="species", how="left", validate="many_to_one")
display(merged[["species", "species_group", "sepal_length", "petal_length"]].head(10))
print("Rows before merge:", len(df), "| Rows after merge:", len(merged))

## 9. Key observations and insights
Use the generated tables and charts to confirm these observations after running the notebook:
1. The dataset contains 150 flower observations and four numeric measurement features, plus a target/species label.
2. Grouped summaries make it possible to compare average sepal length, maximum petal width, and petal-length variability by species.
3. Petal length and petal width generally show a strong positive relationship in this dataset; correlation indicates association, not causation.
4. The box plot and scatter plot help compare species distributions and reveal how measurements separate the species.
5. The cleaning exercise demonstrates how to detect missing values and duplicate records and apply documented fixes.
6. The merge adds a species-group label without changing the number of rows when the lookup keys are unique.

## 10. Conclusion
This practical exercise covered environment verification, loading and inspecting a dataset, Pandas filtering and aggregation, NumPy numerical operations, data cleaning, visualization, descriptive statistics, correlation analysis, and merging. The notebook is organized into independent sections and uses a fixed public sample dataset so the analysis can be reproduced. Before final submission, run all cells, check that every chart and output is visible, add your name and internship dates, and follow any additional format requested by the Yuva internship coordinator.